# DiningCode CSV → MySQL ETL

**실행 순서**

| # | 셀 | 내용 |
|---|----|---------|
| 1 | 설치 & 임포트 | 패키지 설치, 라이브러리 로드 |
| 2 | 설정 | DB 접속 정보, CSV 경로, ENUM 매핑 |
| 3 | DB 연결 & 스키마 초기화 | 테이블 생성 (최초 1회) |
| 4 | 변환 함수 | CSV 정제 및 테이블별 추출 함수 |
| 5 | 적재 함수 | 테이블별 INSERT 함수 |
| 6 | 실행 — 전체 도시 | 4개 도시 순차 적재 |
| 7 | 실행 — 단일 도시 | 특정 도시만 개별 실행 |
| 8 | 검증 쿼리 | 적재 결과 확인 |

## 1. 설치 & 임포트

In [ ]:
# 패키지 설치 (최초 1회)
!pip install mysql-connector-python pandas

In [ ]:
import re
import logging
import sys
from datetime import datetime

import pandas as pd
import mysql.connector
from mysql.connector import Error

# 로깅 설정
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("etl.log", encoding="utf-8"),
    ],
)
logger = logging.getLogger("etl")
print("임포트 완료")

## 2. 설정

> **여기만 수정하면 됩니다** — DB 접속 정보와 CSV 경로를 환경에 맞게 변경하세요.

In [ ]:
# ── DB 접속 정보 ──────────────────────────────────────────────
DB_CONFIG = {
    "host":     "localhost",
    "port":     3306,
    "user":     "your_user",
    "password": "your_password",
    "database": "diningcode",
    "charset":  "utf8mb4",
}

# ── 도시명 → CSV 파일 경로 ────────────────────────────────────
CSV_FILES = {
    "서울": "data/diningcode_data_crawling_seoul_20260125_1542.csv",
    "부산": "data/diningcode_data_crawling_busan_20260125_1542.csv",
    "경기": "data/diningcode_data_crawling_gyeonggi_20260125_1542.csv",
    "대구": "data/diningcode_data_crawling_daegu_20260125_1542.csv",
}

# ── ENUM 값 → TINYINT 매핑 ────────────────────────────────────
TASTE_MAP   = {"맛: 좋음": 3, "맛: 보통": 2, "맛: 부족": 1}
PRICE_MAP   = {"가격: 만족": 3, "가격: 보통": 2, "가격: 불만": 1}
SERVICE_MAP = {"응대: 좋음": 3, "응대: 보통": 2, "응대: 나쁨": 1}

print("설정 로드 완료")

## 3. DB 연결 & 스키마 초기화

테이블이 없을 때만 생성합니다 (`CREATE TABLE IF NOT EXISTS`). **최초 1회만 실행**하면 됩니다.

In [ ]:
class DBConnection:
    """컨텍스트 매니저로 커넥션을 관리합니다.
    
    with DBConnection() as conn:
        cursor = conn.cursor()
        ...
    """
    def __init__(self):
        self._conn = None

    def __enter__(self):
        try:
            self._conn = mysql.connector.connect(**DB_CONFIG)
            logger.info("DB 연결 성공")
            return self._conn
        except Error as e:
            logger.error("DB 연결 실패: %s", e)
            raise

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self._conn and self._conn.is_connected():
            if exc_type:
                self._conn.rollback()
                logger.warning("롤백 처리됨: %s", exc_val)
            self._conn.close()
            logger.info("DB 연결 종료")

print("DBConnection 클래스 정의 완료")

In [ ]:
# DDL 인라인 정의 (schema.sql 없이 노트북 단독 실행 가능)
SCHEMA_SQL = """
CREATE DATABASE IF NOT EXISTS diningcode
    DEFAULT CHARACTER SET utf8mb4
    DEFAULT COLLATE utf8mb4_unicode_ci;

USE diningcode;

CREATE TABLE IF NOT EXISTS cities (
    id          TINYINT UNSIGNED NOT NULL AUTO_INCREMENT,
    name        VARCHAR(20)      NOT NULL COMMENT '서울, 부산, 경기, 대구',
    source_file VARCHAR(100)     NOT NULL COMMENT '원본 CSV 파일명',
    PRIMARY KEY (id),
    UNIQUE KEY uq_city_name (name)
) ENGINE=InnoDB COMMENT='도시 마스터';

CREATE TABLE IF NOT EXISTS areas (
    id      SMALLINT UNSIGNED NOT NULL AUTO_INCREMENT,
    city_id TINYINT UNSIGNED  NOT NULL,
    name    VARCHAR(50)       NOT NULL COMMENT '남대문, 성수 등 item_area',
    PRIMARY KEY (id),
    UNIQUE KEY uq_area (city_id, name),
    CONSTRAINT fk_area_city FOREIGN KEY (city_id) REFERENCES cities (id)
) ENGINE=InnoDB COMMENT='지역 마스터 (도시 하위)';

CREATE TABLE IF NOT EXISTS users (
    id               INT UNSIGNED  NOT NULL AUTO_INCREMENT,
    name             VARCHAR(100)  NOT NULL COMMENT '크롤링된 닉네임',
    tot_avg_rating   DECIMAL(3, 1) DEFAULT NULL,
    tot_rating_num   INT UNSIGNED  DEFAULT 0,
    tot_follow_num   INT UNSIGNED  DEFAULT 0,
    PRIMARY KEY (id),
    UNIQUE KEY uq_user_name (name)
) ENGINE=InnoDB COMMENT='리뷰어 사용자';

CREATE TABLE IF NOT EXISTS restaurants (
    id          INT UNSIGNED      NOT NULL AUTO_INCREMENT,
    city_id     TINYINT UNSIGNED  NOT NULL,
    area_id     SMALLINT UNSIGNED NOT NULL,
    name        VARCHAR(100)      NOT NULL,
    spec_area   VARCHAR(200)      DEFAULT NULL COMMENT '도로명 주소',
    avg_rating  DECIMAL(3, 1)     DEFAULT NULL,
    PRIMARY KEY (id),
    UNIQUE KEY uq_restaurant (city_id, name, spec_area),
    CONSTRAINT fk_rest_city FOREIGN KEY (city_id) REFERENCES cities (id),
    CONSTRAINT fk_rest_area FOREIGN KEY (area_id) REFERENCES areas (id),
    INDEX idx_city   (city_id),
    INDEX idx_area   (area_id),
    INDEX idx_rating (avg_rating)
) ENGINE=InnoDB COMMENT='식당';

CREATE TABLE IF NOT EXISTS reviews (
    id            INT UNSIGNED     NOT NULL AUTO_INCREMENT,
    restaurant_id INT UNSIGNED     NOT NULL,
    user_id       INT UNSIGNED     NOT NULL,
    rating        DECIMAL(2, 1)    DEFAULT NULL COMMENT '1.0 ~ 5.0',
    query         TEXT             DEFAULT NULL COMMENT '리뷰 본문',
    taste         TINYINT UNSIGNED DEFAULT NULL COMMENT '1=부족 2=보통 3=좋음',
    price         TINYINT UNSIGNED DEFAULT NULL COMMENT '1=불만 2=보통 3=만족',
    service       TINYINT UNSIGNED DEFAULT NULL COMMENT '1=나쁨 2=보통 3=좋음',
    menu          TEXT             DEFAULT NULL COMMENT '방문 메뉴',
    review_date   DATE             DEFAULT NULL,
    PRIMARY KEY (id),
    CONSTRAINT fk_rev_restaurant FOREIGN KEY (restaurant_id) REFERENCES restaurants (id),
    CONSTRAINT fk_rev_user       FOREIGN KEY (user_id)       REFERENCES users (id),
    INDEX idx_restaurant (restaurant_id),
    INDEX idx_user       (user_id),
    INDEX idx_date       (review_date)
) ENGINE=InnoDB COMMENT='리뷰';
"""

print("DDL 정의 완료")

In [ ]:
def init_schema():
    """DB 및 테이블을 초기화합니다. 최초 1회만 실행하세요."""
    # DB 생성은 database 파라미터 없이 연결해야 합니다
    init_config = {k: v for k, v in DB_CONFIG.items() if k != "database"}
    conn = mysql.connector.connect(**init_config)
    cursor = conn.cursor()
    statements = [s.strip() for s in SCHEMA_SQL.split(";") if s.strip()]
    for stmt in statements:
        cursor.execute(stmt)
    conn.commit()
    conn.close()
    logger.info("스키마 초기화 완료")

# ▶ 최초 1회 실행
init_schema()

## 4. 변환 함수 (Transform)

CSV 원본 데이터를 정제하고, 테이블별 INSERT용 레코드 리스트로 추출합니다.

In [ ]:
# ── 내부 변환 헬퍼 ─────────────────────────────────────────────

_DATE_PATTERN   = re.compile(r"(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일")
_RATING_PATTERN = re.compile(r"([\d.]+)점")

def _parse_date(raw):
    """'2025년 12월 21일' → datetime.date"""
    if pd.isna(raw):
        return None
    m = _DATE_PATTERN.search(str(raw))
    if not m:
        return None
    try:
        return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3))).date()
    except ValueError:
        return None

def _parse_rating(raw):
    """'4.5점' → 4.5 (float)"""
    if pd.isna(raw):
        return None
    m = _RATING_PATTERN.search(str(raw))
    return float(m.group(1)) if m else None

def _clean_str(raw):
    """앞뒤 공백·줄바꿈 제거, '...더보기' 꼬리 제거"""
    if pd.isna(raw):
        return None
    s = str(raw).strip()
    s = re.sub(r"\s*\.{3}더보기\s*$", "", s).strip()
    return s or None

print("변환 헬퍼 함수 정의 완료")

In [ ]:
def load_csv(path: str, city_name: str) -> pd.DataFrame:
    """CSV를 읽고 기본 정제를 수행한 DataFrame을 반환합니다."""
    df = pd.read_csv(path)
    df["_city"] = city_name

    # 문자열 정제
    df["user_name"]      = df["user_name"].apply(_clean_str)
    df["user_query"]     = df["user_query"].apply(_clean_str)
    df["item_spec_area"] = df["item_spec_area"].apply(_clean_str)

    # 타입 변환
    df["user_rating_val"] = df["user_rating"].apply(_parse_rating)
    df["review_date"]     = df["date"].apply(_parse_date)

    # ENUM → TINYINT
    df["taste_int"]   = df["taste"].map(TASTE_MAP)
    df["price_int"]   = df["price"].map(PRICE_MAP)
    df["service_int"] = df["service"].map(SERVICE_MAP)

    logger.info("[%s] %d행 로드 완료 → %s", city_name, len(df), path)
    return df

print("load_csv 정의 완료")

In [ ]:
def extract_cities(file_map: dict) -> list:
    """도시 마스터 레코드 리스트를 반환합니다."""
    return [{"name": city, "source_file": path} for city, path in file_map.items()]


def extract_areas(df: pd.DataFrame) -> list:
    """(city, area_name) 유니크 목록을 반환합니다."""
    return (
        df[["_city", "item_area"]]
        .drop_duplicates()
        .rename(columns={"_city": "city_name", "item_area": "area_name"})
        .to_dict("records")
    )


def extract_users(df: pd.DataFrame) -> list:
    """user_name 기준 최신 프로필 1건씩 반환합니다 (중복 유저 통합)."""
    return (
        df[["user_name", "user_tot_avg_rating", "user_tot_rating_num", "user_tot_follow_num"]]
        .drop_duplicates(subset=["user_name"], keep="last")
        .rename(columns={
            "user_tot_avg_rating": "tot_avg_rating",
            "user_tot_rating_num": "tot_rating_num",
            "user_tot_follow_num": "tot_follow_num",
        })
        .to_dict("records")
    )


def extract_restaurants(df: pd.DataFrame) -> list:
    """(city, area, name, spec_area) 유니크 식당 목록을 반환합니다."""
    return (
        df[["_city", "item_area", "item_name", "item_spec_area", "item_avg_rating"]]
        .drop_duplicates(subset=["_city", "item_name", "item_spec_area"])
        .rename(columns={
            "_city":           "city_name",
            "item_area":       "area_name",
            "item_name":       "name",
            "item_spec_area":  "spec_area",
            "item_avg_rating": "avg_rating",
        })
        .to_dict("records")
    )


def extract_reviews(df: pd.DataFrame) -> list:
    """리뷰 전체를 반환합니다. FK는 loader에서 조회 후 채웁니다."""
    return (
        df[[
            "_city", "item_name", "item_spec_area",
            "user_name",
            "user_rating_val", "user_query",
            "taste_int", "price_int", "service_int",
            "menu", "review_date",
        ]]
        .rename(columns={
            "_city":           "city_name",
            "item_name":       "restaurant_name",
            "item_spec_area":  "restaurant_spec_area",
            "user_rating_val": "rating",
            "user_query":      "query",
            "taste_int":       "taste",
            "price_int":       "price",
            "service_int":     "service",
        })
        .to_dict("records")
    )

print("extract_* 함수 정의 완료")

## 5. 적재 함수 (Loader)

각 함수가 테이블 하나를 담당합니다. **개별 호출이 가능**하므로 특정 테이블만 재적재할 때도 활용할 수 있습니다.

In [ ]:
# ── ID 조회 캐시 헬퍼 ──────────────────────────────────────────

def _fetch_city_ids(cursor) -> dict:
    cursor.execute("SELECT id, name FROM cities")
    return {row[1]: row[0] for row in cursor.fetchall()}

def _fetch_area_ids(cursor) -> dict:
    cursor.execute("SELECT id, city_id, name FROM areas")
    return {(row[1], row[2]): row[0] for row in cursor.fetchall()}

def _fetch_user_ids(cursor) -> dict:
    cursor.execute("SELECT id, name FROM users")
    return {row[1]: row[0] for row in cursor.fetchall()}

def _fetch_restaurant_ids(cursor) -> dict:
    cursor.execute("SELECT id, city_id, name, spec_area FROM restaurants")
    return {(row[1], row[2], row[3]): row[0] for row in cursor.fetchall()}

print("ID 조회 헬퍼 정의 완료")

In [ ]:
def load_cities(conn, cities: list) -> dict:
    """
    cities 테이블에 적재합니다.
    반환: {"서울": 1, "부산": 2, ...}
    """
    sql = """
        INSERT INTO cities (name, source_file)
        VALUES (%(name)s, %(source_file)s)
        ON DUPLICATE KEY UPDATE source_file = VALUES(source_file)
    """
    cursor = conn.cursor()
    cursor.executemany(sql, cities)
    conn.commit()
    logger.info("cities 적재 완료: %d건", len(cities))
    return _fetch_city_ids(cursor)

print("load_cities 정의 완료")

In [ ]:
def load_areas(conn, areas: list, city_id_map: dict) -> dict:
    """
    areas 테이블에 적재합니다.
    반환: {(city_id, area_name): area_id, ...}
    """
    rows = [
        {"city_id": city_id_map[r["city_name"]], "name": r["area_name"]}
        for r in areas
        if r["city_name"] in city_id_map and r["area_name"]
    ]
    sql = """
        INSERT INTO areas (city_id, name)
        VALUES (%(city_id)s, %(name)s)
        ON DUPLICATE KEY UPDATE name = VALUES(name)
    """
    cursor = conn.cursor()
    cursor.executemany(sql, rows)
    conn.commit()
    logger.info("areas 적재 완료: %d건", len(rows))
    return _fetch_area_ids(cursor)

print("load_areas 정의 완료")

In [ ]:
def load_users(conn, users: list) -> dict:
    """
    users 테이블에 적재합니다.
    반환: {"닉네임": user_id, ...}
    """
    rows = [r for r in users if r.get("user_name")]
    sql = """
        INSERT INTO users (name, tot_avg_rating, tot_rating_num, tot_follow_num)
        VALUES (%(user_name)s, %(tot_avg_rating)s, %(tot_rating_num)s, %(tot_follow_num)s)
        ON DUPLICATE KEY UPDATE
            tot_avg_rating = VALUES(tot_avg_rating),
            tot_rating_num = VALUES(tot_rating_num),
            tot_follow_num = VALUES(tot_follow_num)
    """
    cursor = conn.cursor()
    cursor.executemany(sql, rows)
    conn.commit()
    logger.info("users 적재 완료: %d건", len(rows))
    return _fetch_user_ids(cursor)

print("load_users 정의 완료")

In [ ]:
def load_restaurants(conn, restaurants: list, city_id_map: dict, area_id_map: dict) -> dict:
    """
    restaurants 테이블에 적재합니다.
    반환: {(city_id, name, spec_area): restaurant_id, ...}
    """
    rows = []
    for r in restaurants:
        city_id = city_id_map.get(r["city_name"])
        area_id = area_id_map.get((city_id, r["area_name"]))
        if city_id is None or area_id is None:
            logger.warning("city/area 매핑 실패 → 스킵: %s / %s", r["city_name"], r["area_name"])
            continue
        rows.append({
            "city_id":    city_id,
            "area_id":    area_id,
            "name":       r["name"],
            "spec_area":  r.get("spec_area"),
            "avg_rating": r.get("avg_rating"),
        })

    sql = """
        INSERT INTO restaurants (city_id, area_id, name, spec_area, avg_rating)
        VALUES (%(city_id)s, %(area_id)s, %(name)s, %(spec_area)s, %(avg_rating)s)
        ON DUPLICATE KEY UPDATE
            avg_rating = VALUES(avg_rating),
            area_id    = VALUES(area_id)
    """
    cursor = conn.cursor()
    cursor.executemany(sql, rows)
    conn.commit()
    logger.info("restaurants 적재 완료: %d건", len(rows))
    return _fetch_restaurant_ids(cursor)

print("load_restaurants 정의 완료")

In [ ]:
def load_reviews(conn, reviews: list, city_id_map: dict,
                 restaurant_id_map: dict, user_id_map: dict,
                 batch_size: int = 500) -> None:
    """
    reviews 테이블에 적재합니다.
    FK(restaurant_id, user_id)를 여기서 조회·매핑 후 INSERT합니다.
    batch_size 단위로 나눠 INSERT하여 메모리를 제어합니다.
    """
    sql = """
        INSERT IGNORE INTO reviews
            (restaurant_id, user_id, rating, query, taste, price, service, menu, review_date)
        VALUES
            (%(restaurant_id)s, %(user_id)s, %(rating)s, %(query)s,
             %(taste)s, %(price)s, %(service)s, %(menu)s, %(review_date)s)
    """
    rows, skipped = [], 0

    for r in reviews:
        city_id       = city_id_map.get(r["city_name"])
        restaurant_id = restaurant_id_map.get((city_id, r["restaurant_name"], r["restaurant_spec_area"]))
        user_id       = user_id_map.get(r["user_name"])

        if restaurant_id is None or user_id is None:
            skipped += 1
            continue

        rows.append({
            "restaurant_id": restaurant_id,
            "user_id":       user_id,
            "rating":        r.get("rating"),
            "query":         r.get("query"),
            "taste":         r.get("taste"),
            "price":         r.get("price"),
            "service":       r.get("service"),
            "menu":          r.get("menu"),
            "review_date":   r.get("review_date"),
        })

    cursor = conn.cursor()
    total = 0
    for i in range(0, len(rows), batch_size):
        batch = rows[i : i + batch_size]
        cursor.executemany(sql, batch)
        conn.commit()
        total += len(batch)

    logger.info("reviews 적재 완료: %d건 (스킵: %d건)", total, skipped)

print("load_reviews 정의 완료")

## 6. 실행 — 전체 도시 적재

4개 도시 CSV를 순차적으로 처리합니다. 한 도시에서 오류가 발생해도 나머지 도시는 계속 진행됩니다.

In [ ]:
def run_city(conn, city_name: str, csv_path: str, city_id_map: dict, skip_reviews: bool = False):
    """단일 도시에 대한 전체 ETL을 수행합니다."""
    logger.info("══ [%s] ETL 시작 ══", city_name)

    df = load_csv(csv_path, city_name)

    area_id_map       = load_areas(conn, extract_areas(df), city_id_map)
    user_id_map       = load_users(conn, extract_users(df))
    restaurant_id_map = load_restaurants(conn, extract_restaurants(df), city_id_map, area_id_map)

    if not skip_reviews:
        load_reviews(conn, extract_reviews(df), city_id_map, restaurant_id_map, user_id_map)

    logger.info("══ [%s] ETL 완료 ══", city_name)

print("run_city 정의 완료")

In [ ]:
# ▶ 전체 도시 적재 실행
TARGET_CITIES = CSV_FILES  # 모든 도시 (일부만 하려면 아래 셀 7 사용)

with DBConnection() as conn:
    # 1. cities 먼저 적재 (FK 기준)
    city_id_map = load_cities(conn, extract_cities(TARGET_CITIES))

    # 2. 도시별 순차 처리
    for city_name, csv_path in TARGET_CITIES.items():
        try:
            run_city(conn, city_name, csv_path, city_id_map)
        except Exception as e:
            logger.error("[%s] 오류 발생 — 스킵: %s", city_name, e, exc_info=True)

print("전체 ETL 완료")

## 7. 실행 — 단일 도시 적재

특정 도시만 개별적으로 적재하거나 재실행할 때 사용합니다.

In [ ]:
# ▶ 적재할 도시를 직접 지정
CITY_NAME = "서울"  # 변경: 서울 / 부산 / 경기 / 대구
SKIP_REVIEWS = False  # True: 마스터 데이터만 적재 (식당·유저)

with DBConnection() as conn:
    city_id_map = load_cities(conn, extract_cities({CITY_NAME: CSV_FILES[CITY_NAME]}))
    run_city(conn, CITY_NAME, CSV_FILES[CITY_NAME], city_id_map, skip_reviews=SKIP_REVIEWS)

print(f"{CITY_NAME} 적재 완료")

## 8. 검증 쿼리

적재 결과를 확인합니다.

In [ ]:
VERIFY_QUERIES = {
    "테이블별 건수": """
        SELECT 'cities'      AS tbl, COUNT(*) AS cnt FROM cities
        UNION ALL
        SELECT 'areas',               COUNT(*)        FROM areas
        UNION ALL
        SELECT 'users',               COUNT(*)        FROM users
        UNION ALL
        SELECT 'restaurants',         COUNT(*)        FROM restaurants
        UNION ALL
        SELECT 'reviews',             COUNT(*)        FROM reviews
    """,
    "도시별 식당 수": """
        SELECT c.name AS city, COUNT(r.id) AS restaurant_cnt
        FROM cities c
        LEFT JOIN restaurants r ON r.city_id = c.id
        GROUP BY c.name
    """,
    "도시별 리뷰 수": """
        SELECT c.name AS city, COUNT(rv.id) AS review_cnt
        FROM cities c
        LEFT JOIN restaurants r  ON r.city_id  = c.id
        LEFT JOIN reviews     rv ON rv.restaurant_id = r.id
        GROUP BY c.name
    """,
}

with DBConnection() as conn:
    cursor = conn.cursor()
    for title, sql in VERIFY_QUERIES.items():
        cursor.execute(sql)
        rows = cursor.fetchall()
        cols = [desc[0] for desc in cursor.description]
        df_result = pd.DataFrame(rows, columns=cols)
        print(f"\n── {title} ──")
        display(df_result)